# Prepare Notebook Environment for Strands Agents

This section prepares your notebook environment with the dependencies needed to create agents using the Strands framework.

# Prerequisites

1. If you are running the notebook in your own account not as part of an AWS hosted event, follow the **deployment** instructions in the [Self Paced](https://catalog.us-east-1.prod.workshops.aws/workshops/0bf501fc-357d-43b3-b758-b44f9b0e7d49/en-US/introduction/access-and-navigate-aws-account/self-paced) workshop to set up the following dependencies in your account:
    - Networking infrastructure (VPC, subnets, etc.)
    - Amazon Redshift database
    - Amazon Bedrock Knowledge Base
2. Configure Bedrock model access for the following models:
    - Amazon
        - Titan Embeddings G1 - Text
    - Anthropic
        - Anthropic Claude 3.7 Sonnet
3. Ensure your notebook execution role has the following managed policies:
    - AmazonBedrockFullAccess
    - AmazonRedshiftQueryEditor
    - AmazonS3FullAccess
    - AmazonSageMakerFullAccess
    - AWSLambda_FullAccess
    - AWSStepFunctionsFullAccess
    - IAMFullAccess
    - AWSCodeBuildAdminAccess
4. For the protein design notebook, additional AWS HealthOmics permissions will be automatically added by this setup notebook.

# Environment Setup

#### Run the pip command below to install all required packages

In [ ]:
%pip install boto3 awscli botocore termcolor sagemaker pytrials streamlit --quiet

Verify that the boto3 version shown below is **1.37.1** or higher.

In [ ]:
%pip show boto3

#### Import Python libraries

In [ ]:
# Standard Python libraries
import boto3
import sagemaker

# Import needed functions to create agent
from utils.bedrock_agent_helper import AgentsForAmazonBedrock
from utils.role_policy_helper import SageMakerRolePolicyChecker

#### Extract SageMaker role and account information needed for agent creation

In [ ]:
# boto3 session
sts_client = boto3.client('sts')
session = boto3.session.Session()

# Account information
account_id = sts_client.get_caller_identity()["Account"]
region = session.region_name
print(f"Account ID: {account_id}")
print(f"Region: {region}")

# Get SageMaker session and execution role
sagemaker_session = sagemaker.Session()
role = sagemaker_session.get_caller_identity_arn()
print(f"SageMaker Execution Role: {role}")

#### Verify that the SageMaker role has the necessary policies for notebook execution

In [ ]:
# Check and add required policies
role_checker = SageMakerRolePolicyChecker()
role_name = role.split('/')[-1]  # Extract role name from ARN

try:
    # Check existing policies
    role_checker.check_policies(role)
    print("All required managed policies are attached!")
except Exception as e:
    print(f"Missing policies: {e}")
    print("Please manually attach the missing policies or contact your administrator.")



#### Test permissions for key AWS services used in the workshop

In [ ]:
# Test permissions for key AWS services
from termcolor import colored
import json

def test_service_permission(client_name, test_operation, operation_params=None):
    """Helper function to test service permissions"""
    try:
        client = boto3.client(client_name, region_name=region)
        
        # Execute test operation
        if operation_params:
            response = getattr(client, test_operation)(**operation_params)
        else:
            response = getattr(client, test_operation)()
            
        return True, "Permission verified"
    except client.exceptions.ResourceNotFoundException:
        # Resource not found is not a permission issue
        return True, "Permission verified (resource not found)"
    except Exception as e:
        error_msg = str(e)
        if "AccessDenied" in error_msg or "UnauthorizedOperation" in error_msg:
            return False, f"Permission denied: {error_msg[:100]}..."
        elif "ResourceNotFound" in error_msg or "does not exist" in error_msg:
            return True, "Permission verified (resource not found)"
        else:
            # Other errors may not be permission-related
            return True, f"Permission verified (other: {error_msg[:50]}...)"

# List of services to test
services_to_test = [
    {
        "name": "Amazon Bedrock (Knowledge Base)",
        "client": "bedrock-agent",
        "operation": "list_knowledge_bases",
        "params": {"maxResults": 1}
    },
    {
        "name": "Amazon Bedrock (Agents/AgentCore)",
        "client": "bedrock-agent",
        "operation": "list_agents",
        "params": {"maxResults": 1}
    },
    {
        "name": "Amazon Bedrock (Models)",
        "client": "bedrock",
        "operation": "list_foundation_models",
        "params": None
    },
    {
        "name": "Amazon S3",
        "client": "s3",
        "operation": "list_buckets",
        "params": None
    },
    {
        "name": "AWS Lambda",
        "client": "lambda",
        "operation": "list_functions",
        "params": {"MaxItems": 1}
    },
    {
        "name": "Amazon Redshift",
        "client": "redshift",
        "operation": "describe_clusters",
        "params": None
    }
]

print("=" * 60)
print("AWS Service Permission Test Results")
print("=" * 60)

all_passed = True
for service in services_to_test:
    success, message = test_service_permission(
        service["client"],
        service["operation"],
        service["params"]
    )
    
    if success:
        status = colored("✓ Pass", "green")
    else:
        status = colored("✗ Fail", "red")
        all_passed = False
    
    print(f"{status} - {service['name']}")
    if not success:
        print(f"     {message}")

print("=" * 60)

if all_passed:
    print(colored("✓ All service permission tests passed!", "green"))
    print("Ready to proceed with the workshop.")
else:
    print(colored("⚠ Some permissions are missing.", "yellow"))
    print("Please verify and add permissions for the failed services above.")